# Feature engineering — is a grassland-labeled pixel actually forest?

Goal: assemble a model-ready table where each Florida pixel that LANDFIRE EVT calls **grassland / ag / shrub** gets a rich feature vector and a label for **"actually recently clearcut forest."** LCMS tree-removal almost never fires inside those EVT classes, so labels are **anchor-based / semi-supervised**:

- **positive_forest** — confident clearcuts (LCMS pre-Trees → Tree Removal).
- **negative_grassland** — stable genuine non-forest (never Trees in LCMS history, not forest in EVT-2016, not a clearcut).
- **apply_confused** — the three confused EVT classes (7997/9823/9585); the model's inference target, scored but not used for training.

**Features** (see `cac.feature_dictionary()`): AlphaEarth event + pre-year embeddings (128), their L2 delta (disturbance magnitude), LCMS multi-year history, and EVT-change. Each column is tagged with `lcms_derived`; since the label is LCMS-derived, the leakage-free predictor set is embeddings + EVT (`cac.clean_feature_columns()`).

Builds on `notebooks/clearcut_ag_common.py`; upstream analysis in the two `Clearcut-vs-Agriculture-*` notebooks. Design: `docs/superpowers/specs/2026-07-01-clearcut-vs-agriculture-embeddings-design.md`.

## 1. Setup

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd()
_candidates = [cwd / "notebooks", cwd, *[p / "notebooks" for p in cwd.parents], *cwd.parents]
nb_dir = next(p for p in _candidates if (p / "clearcut_ag_common.py").exists())
sys.path.insert(0, str(nb_dir))

import clearcut_ag_common as cac
import geopandas as gpd
import numpy as np
import pandas as pd

REPO_ROOT = cac.find_repo_root(nb_dir)
ee = cac.init_ee()
florida, florida_gdf = cac.load_florida(REPO_ROOT)
florida_gdf_5070 = gpd.read_file(REPO_ROOT / "config" / "extent.geojson").to_crs("EPSG:5070")
OUT_DIR = REPO_ROOT / "data" / "interim" / "clearcut_ag"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)

Repo root: /home/chazm/projects/artemis-model


## 2. Parameters

In [2]:
EVENT_YEAR = cac.DEFAULT_EVENT_YEAR   # 2022
PRE_YEAR = cac.DEFAULT_PRE_YEAR       # 2020
N_PER_CONFUSED = 250      # per confused class (guarantees floodplain shrubland)
N_AGRICULTURE = 300       # row-crop / agricultural EVT (clean negatives)
N_GRASS_SHRUB_OTHER = 300 # other herb/shrub EVT (clean negatives)
N_CLEARCUT = 400          # LCMS clearcut mask (positives)

## 3. Build the feature table
Stratified EVT sampling (two-stage decimated → full-res, so rare floodplain shrubland is captured) plus LCMS clearcut points, each attributed with the full feature stack.

In [3]:
features, feat_dict = cac.build_feature_table(
    florida, florida_gdf_5070, REPO_ROOT,
    event_year=EVENT_YEAR, pre_year=PRE_YEAR,
    n_per_confused=N_PER_CONFUSED, n_agriculture=N_AGRICULTURE,
    n_grass_shrub_other=N_GRASS_SHRUB_OTHER, n_clearcut=N_CLEARCUT,
)
features = features.dropna(subset=list(cac.EMBEDDING_BANDS)).reset_index(drop=True)
print("rows:", len(features), "cols:", features.shape[1])
print("\nrole:\n", features.role.value_counts())
print("\nlabel y:\n", features.y.value_counts(dropna=False))
print("\nsampling stratum:\n", features.stratum.value_counts())

rows: 1750 cols: 156

role:
 role
apply_confused        750
positive_forest       400
other                 319
negative_grassland    281
Name: count, dtype: int64

label y:
 y
NaN    1069
1.0     400
0.0     281
Name: count, dtype: int64

sampling stratum:
 stratum
lcms_clearcut          400
agriculture_rowcrop    300
grass_shrub_other      300
pasture_hay            250
ruderal_grass          250
floodplain_shrub       250
Name: count, dtype: int64


## 4. Sanity checks

In [4]:
print("NaN in event embeddings:", int(features[list(cac.EMBEDDING_BANDS)].isna().sum().sum()))
print("NaN in pre embeddings:  ", int(features[[f"P{i:02d}" for i in range(64)]].isna().sum().sum()))
conf = features[features.stratum.isin(["pasture_hay","ruderal_grass","floodplain_shrub"])]
print("confused sampling class-match rate:", round(float((conf.confused_name==conf.stratum).mean()),3))
# embedding disturbance magnitude should be larger for clearcuts
print("\nemb_delta_l2 by role:\n", features.groupby("role").emb_delta_l2.describe()[["count","mean","50%"]].round(3))

NaN in event embeddings: 0
NaN in pre embeddings:   0
confused sampling class-match rate: 1.0

emb_delta_l2 by role:
                     count   mean    50%
role                                   
apply_confused      750.0  0.313  0.293
negative_grassland  281.0  0.399  0.334
other               319.0  0.321  0.267
positive_forest     400.0  0.719  0.727


## 5. Feature families and leakage
The data dictionary tags every column. Because the label is derived from LCMS, only `lcms_derived == False` columns are safe predictors for the honest model.

In [5]:
print(feat_dict.groupby(["family","lcms_derived"]).size())
clean = cac.clean_feature_columns()
print("\nclean (leakage-free) predictor count:", len(clean))
print("example clean non-embedding cols:", [c for c in clean if not c.startswith(("A","P"))])

family           lcms_derived
embedding_delta  False            1
embedding_event  False           64
embedding_pre    False           64
evt              False            3
lcms             True             6
dtype: int64

clean (leakage-free) predictor count: 132
example clean non-embedding cols: ['emb_delta_l2', 'is_forest_2016', 'evt_change_strict', 'evt_change_broad']


## 6. Baseline model with spatial cross-validation
Train on the anchors (positive_forest vs negative_grassland) using only leakage-free features, evaluated under spatial block CV (GroupKFold by 0.5° cell). Three feature sets show the value added by the engineered features.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict, GroupKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score

anchors = features[features.role.isin(["positive_forest","negative_grassland"])].copy()
anchors["cell"] = (np.floor(anchors.lon*2).astype(int).astype(str) + "_"
                   + np.floor(anchors.lat*2).astype(int).astype(str))
y = anchors.y.astype(int).to_numpy()
groups = anchors.cell.to_numpy()
gkf = GroupKFold(n_splits=min(5, anchors.cell.nunique()))

event_bands = list(cac.EMBEDDING_BANDS)
pre_bands = [f"P{i:02d}" for i in range(64)]
evt_feats = ["is_forest_2016","evt_change_strict","evt_change_broad"]
featsets = {
    "A: event embeddings (64)": event_bands,
    "B: + pre + delta (129)": event_bands + pre_bands + ["emb_delta_l2"],
    "C: + EVT (clean full)": event_bands + pre_bands + ["emb_delta_l2"] + evt_feats,
}
print(f"anchors: {len(anchors)}  positives(forest)={int(y.sum())}  negatives={int((1-y).sum())}  cells={anchors.cell.nunique()}\n")
for name, cols in featsets.items():
    X = anchors[cols].astype(float).to_numpy()
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    prob = cross_val_predict(clf, X, y, cv=gkf, groups=groups, method="predict_proba")[:,1]
    pred = (prob > 0.5).astype(int)
    print(f"{name:28s} logreg  spatialCV AUC={roc_auc_score(y,prob):.3f}  "
          f"bal_acc={balanced_accuracy_score(y,pred):.3f}  acc={accuracy_score(y,pred):.3f}")

X = anchors[featsets["C: + EVT (clean full)"]].astype(float).to_numpy()
rf = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1)
prob_rf = cross_val_predict(rf, X, y, cv=gkf, groups=groups, method="predict_proba")[:,1]
print(f"\n{'C (random forest)':28s}         spatialCV AUC={roc_auc_score(y,prob_rf):.3f}  "
      f"bal_acc={balanced_accuracy_score(y,(prob_rf>0.5).astype(int)):.3f}")

anchors: 681  positives(forest)=400  negatives=281  cells=71

A: event embeddings (64)     logreg  spatialCV AUC=0.994  bal_acc=0.951  acc=0.953
B: + pre + delta (129)       logreg  spatialCV AUC=1.000  bal_acc=0.998  acc=0.997
C: + EVT (clean full)        logreg  spatialCV AUC=1.000  bal_acc=0.998  acc=0.997



C (random forest)                    spatialCV AUC=1.000  bal_acc=0.990


## 7. Apply the model to the confused classes
Fit feature set C on all anchors and score the apply set. `mean_prob` / `frac_forest` estimate how much of each EVT class is actually clearcut forest.

In [7]:
cols = featsets["C: + EVT (clean full)"]
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(
    anchors[cols].astype(float).to_numpy(), y)
apply_set = features[features.role == "apply_confused"].copy()
apply_set["p_forest"] = model.predict_proba(apply_set[cols].astype(float).to_numpy())[:,1]
summary = (apply_set.groupby("confused_name")
           .agg(n=("p_forest","size"), mean_p_forest=("p_forest","mean"),
                frac_forest=("p_forest", lambda s: float((s>0.5).mean())))
           .round(3))
print(summary)

                    n  mean_p_forest  frac_forest
confused_name                                    
floodplain_shrub  250          0.814        0.844
pasture_hay       250          0.077        0.064
ruderal_grass     250          0.409        0.400


## 8. Persist the feature table and dictionary

In [8]:
features.to_csv(OUT_DIR / "feature_table.csv", index=False)
feat_dict.to_csv(OUT_DIR / "feature_dictionary.csv", index=False)
apply_set.to_csv(OUT_DIR / "confused_scored_full_features.csv", index=False)
print("Wrote feature_table.csv (%d rows, %d cols), feature_dictionary.csv, confused_scored_full_features.csv"
      % (len(features), features.shape[1]))

Wrote feature_table.csv (1750 rows, 156 cols), feature_dictionary.csv, confused_scored_full_features.csv


## Interpretation & next steps

**Read the CV numbers carefully — AUC ≈ 1.0 is largely by construction, not a hard win.**
Positives are *defined* as pre-forest (LCMS Trees in `PRE_YEAR`) and negatives as never-forest,
and the **pre-year embedding encodes exactly "was forest in `PRE_YEAR`."** So feature sets B/C
(which include the pre-year embedding + delta) separate the anchors almost perfectly because the
strongest feature is near-circular with the label. Treat B/C as an **upper bound**, not evidence
of a hard prediction solved.

- **Feature set A (event-year embedding only, AUC 0.994)** is the honest *appearance-based*
  number: a recent clearcut (bare soil / slash / early regrowth) looks different from established
  grassland even without temporal context.
- The **A → B lift** shows temporal features (pre-year embedding, `emb_delta_l2`) add real signal
  — corroborated by `emb_delta_l2` being ~0.72 for clearcuts vs ~0.40 for stable grassland — but
  because that signal overlaps the label definition, its CV score is inflated.
- The **apply-set `frac_forest` per confused class is the actual deliverable**: floodplain
  shrubland ≈ 0.81 (mostly actually-forest), ruderal grassland ≈ 0.41 (mixed), pasture/hay ≈ 0.06
  (mostly genuine agriculture). Note this differs from the embeddings-only Method-1 ranking
  (ruderal highest) because the temporal features re-weight toward "was it forest 2 years ago."
  These are **estimates that need imagery/field validation**, not ground truth.

**To make the evaluation honest (recommended next):**
- Decouple label from features: define positives using an *earlier* pre-year than the embedding
  pre-year, or hold out the pre-year embedding when scoring anchor performance.
- Validate the apply-set predictions against high-resolution imagery (NAIP) or hand labels to turn
  this from semi-supervised into a supervised, trustworthy classifier.

**Easy feature adds** (all already in the repo): forest ownership (RDS-2025-0045; industrial
timberland raises clearcut prior), 3DEP terrain, POLARIS soils, PRISM climate — append as columns
in `sample_features`.
